# 전처리 연습 (Tokenize, Cleansing & Normalization, Stemming & Lemmatization)

1. 데이터셋을 찾는다. (만들어진 데이터셋, 크롤링, ...)
    - **이어지는 실습에 쓸 수 있을 법한 데이터셋 찾기 Tip**
    - 소설 등 이어지는 긴 문장 여러 건 ok
    - 대화형 데이터셋도 good
        - QnA로 구성되어있는 corpus도 좋고
        - 일반적인 대화로 구성되어있는 corpus도 좋음!
2. 전처리

    2-1. 의미가 있는 단어 단위로 vocabulary -> 단어로만 구성된 데이터셋을 가지고 있으면 안 됨
    
    2-2. corpus -> 토큰화 -> 전처리 -> 문장(불필요한 특수문자, 이모지, 링크나 이미지에 대한 소스가 포함되어있는지)
    -> 종류 뭐든 상관없지만 거거익선

In [10]:
import zipfile, json, pandas as pd

zip_path = r"C://SKN 24_Yunu Jeon//NLP_practice//08.전문 의학지식 데이터//3.개방데이터//1.데이터//Training//02.라벨링데이터//TL_정신건강의학과.zip" 

rows = []
with zipfile.ZipFile(zip_path, "r") as z:
    json_names = [n for n in z.namelist() if n.lower().endswith(".json")]
    
    for name in json_names[:1563]:
        with z.open(name) as f:
            text = f.read().decode("utf-8-sig")
            rows.append(json.loads(text))

df = pd.DataFrame(rows)
df.head()

,qa_id,domain,q_type,question,answer
0,236,3,1,환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,5) 모델링(Modeling)
1,244,3,1,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그...",2) 투렛장애
2,243,3,1,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ...",4) 인지행동치료
3,4441,3,1,35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,4) 전두엽 (frontal lobe)
4,237,3,2,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발...",고전적 조건 형성


In [11]:
df.to_csv('output.csv', index=False, encoding='utf-8-sig')

In [2]:
df = df[['question', 'answer']]
df.head()

,question,answer
0,환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,5) 모델링(Modeling)
1,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그...",2) 투렛장애
2,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ...",4) 인지행동치료
3,35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,4) 전두엽 (frontal lobe)
4,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발...",고전적 조건 형성


# 1. question 처리

### 1-1. 특수문자 처리

In [3]:
import re

def clean_question(text):
    text = str(text)
    
    # 1. 공백 정리
    text = re.sub(r"\s+", " ", text)
    
    # 2. 한글, 영문, 숫자, 기본 문장 부호만 남기고 없애기
    text = re.sub(r"[^0-9A-Za-z가-힣\s\.\,\?\!]", " ", text)
    
    # 3. 공백 정리 다시!
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df['question_clean'] = df['question'].apply(clean_question)
df.head()

,question,answer,question_clean
0,환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,5) 모델링(Modeling),환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...
1,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그...",2) 투렛장애,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그..."
2,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ...",4) 인지행동치료,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ..."
3,35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,4) 전두엽 (frontal lobe),35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...
4,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발...",고전적 조건 형성,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발..."


### 1-2. 토크나이징

In [4]:
# 토크나이징 시 10, 세 등으로 분리되는 문제가 발생하여 이 부분을 해결함

import re
from konlpy.tag import Okt

okt = Okt()

# 1. 기본 텍스트 정제: 숫자 포함, 불필요 특수문자만 제거
def clean_question(text: str) -> str:
    text = str(text)

    # 유니코드 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    # (선택) 이상한 제어문자 제거
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)  # zero-width, BOM 등

    # 숫자/영문/한글/기본문장부호만 남김 (콤마는 일단 남겨둠: 아래에서 숫자-단위 결합에 쓰고 제거)
    text = re.sub(r"[^0-9A-Za-z가-힣\s\.\,\?\!\%\(\)\-\/]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


# 2. Okt 전에 정규화: 콤마 통일 + 숫자/단위 결합
def normalize_before_okt(text: str) -> str:
    text = str(text)

    # 콤마류 통일
    text = re.sub(r"[，､﹐﹑]", ",", text)

    # "10, 세" / "10,세" / "10 , 세" -> "10세"
    text = re.sub(r"(\d+)\s*,\s*(세|개월|년|일|시간|분)", r"\1\2", text)

    # "10 세" -> "10세" (단위 확장 가능)
    text = re.sub(
        r"(\d+)\s+(세|개월|년|일|시간|분|mg|g|kg|ml|l|mm|cm|m|%)",
        r"\1\2",
        text,
        flags=re.I
    )

    # 정규화 끝났으니 콤마는 일반 문장부호로서 제거해도 됨(토큰 깨짐 방지)
    text = text.replace(",", " ")

    text = re.sub(r"\s+", " ", text).strip()
    return text


# 3) Okt가 "10세"를 "10"+"세"로 쪼개도, 토큰에서 다시 붙이기
def merge_num_unit_tokens(tokens, units=('세','개월','년','일','시간','분','mg','g','kg','ml','l','mm','cm','m','%')):
    merged = []
    i = 0
    while i < len(tokens):
        if tokens[i].isdigit() and i + 1 < len(tokens) and tokens[i+1] in units:
            merged.append(tokens[i] + tokens[i+1])  # 10 + 세 -> 10세
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged


# 4) 최종 토크나이저
def tokenize_okt(text: str):
    text = clean_question(text)
    text = normalize_before_okt(text)

    toks = okt.morphs(text)
    toks = merge_num_unit_tokens(toks)

    return toks


# 적용: question 컬럼 기준
df["question_clean"] = df["question"].apply(clean_question)
df["question_norm"]  = df["question_clean"].apply(normalize_before_okt)
df["tokens"] = df["question"].apply(tokenize_okt)

In [5]:
stopwords = set(['은','는','이','가','을','를','에','에서','으로','로','와','과','도','만'])

df['questions_cleaned'] = df['tokens'].apply(
    lambda toks: [t for t in toks if t not in stopwords]
)

df.head()

,question,answer,question_clean,question_norm,tokens,questions_cleaned
0,환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,5) 모델링(Modeling),환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,"[환자, 가, 특정, 행동, 을, 관찰, 하고, 이를, 모방, 하, 거나, 흉내, ...","[환자, 특정, 행동, 관찰, 하고, 이를, 모방, 하, 거나, 흉내, 내는, 과정..."
1,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그...",2) 투렛장애,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그...",10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고 얼굴을 찡그리...,"[10세, 남아, 가, 약, 2년, 전, 부터, 눈, 을, 자주, 깜박, 거리, 고...","[10세, 남아, 약, 2년, 전, 부터, 눈, 자주, 깜박, 거리, 고, 코, 주..."
2,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ...",4) 인지행동치료,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ...",49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울 불안 무기력 등의 증상...,"[49세, 여성, 이, 5년, 이상, 불면증, 으로, 내, 원하였다, ., 환자, ...","[49세, 여성, 5년, 이상, 불면증, 내, 원하였다, ., 환자, 우울, 불안,..."
3,35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,4) 전두엽 (frontal lobe),35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,"[35세, 남자, 가, 갑작스러운, 성격, 변화, 와, 잦은, 돌발, 행동, 으로,...","[35세, 남자, 갑작스러운, 성격, 변화, 잦은, 돌발, 행동, 병원, 왔다, ...."
4,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발...",고전적 조건 형성,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발...",중립 자극이 무조건 자극과 반복적으로 연합되어 중립 자극만으로도 특정 반응을 유발하...,"[중립, 자극, 이, 무조건, 자극, 과, 반복, 적, 으로, 연합, 되어, 중립,...","[중립, 자극, 무조건, 자극, 반복, 적, 연합, 되어, 중립, 자극, 만으로도,..."


# 2. Answer

In [6]:
# 기본 정규화
import re

def clean_label(label):
    label = str(label)

    # 1) 앞 숫자 제거 (예: 2) )
    label = re.sub(r"^\d+\)\s*", "", label)

    # 2) 괄호 안 영어 제거
    label = re.sub(r"\s*\(.*?\)", "", label)

    # 3) 공백 정리
    label = re.sub(r"\s+", " ", label).strip()

    return label

df['answer_label'] = df['answer'].apply(clean_label)
df.head()

,question,answer,question_clean,question_norm,tokens,questions_cleaned,answer_label
0,환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,5) 모델링(Modeling),환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,환자가 특정 행동을 관찰하고 이를 모방하거나 흉내 내는 과정을 통해 학습이 이루어진...,"[환자, 가, 특정, 행동, 을, 관찰, 하고, 이를, 모방, 하, 거나, 흉내, ...","[환자, 특정, 행동, 관찰, 하고, 이를, 모방, 하, 거나, 흉내, 내는, 과정...",모델링
1,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그...",2) 투렛장애,"10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고, 얼굴을 찡그...",10세 남아가 약 2년 전부터 눈을 자주 깜박거리고 코에 주름을 잡고 얼굴을 찡그리...,"[10세, 남아, 가, 약, 2년, 전, 부터, 눈, 을, 자주, 깜박, 거리, 고...","[10세, 남아, 약, 2년, 전, 부터, 눈, 자주, 깜박, 거리, 고, 코, 주...",투렛장애
2,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ...",4) 인지행동치료,"49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울, 불안, 무기력 등의 ...",49세 여성이 5년 이상 불면증으로 내원하였다. 환자는 우울 불안 무기력 등의 증상...,"[49세, 여성, 이, 5년, 이상, 불면증, 으로, 내, 원하였다, ., 환자, ...","[49세, 여성, 5년, 이상, 불면증, 내, 원하였다, ., 환자, 우울, 불안,...",인지행동치료
3,35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,4) 전두엽 (frontal lobe),35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,35세 남자가 갑작스러운 성격 변화와 잦은 돌발 행동으로 병원에 왔다. 평소 차분한...,"[35세, 남자, 가, 갑작스러운, 성격, 변화, 와, 잦은, 돌발, 행동, 으로,...","[35세, 남자, 갑작스러운, 성격, 변화, 잦은, 돌발, 행동, 병원, 왔다, ....",전두엽
4,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발...",고전적 조건 형성,"중립 자극이 무조건 자극과 반복적으로 연합되어, 중립 자극만으로도 특정 반응을 유발...",중립 자극이 무조건 자극과 반복적으로 연합되어 중립 자극만으로도 특정 반응을 유발하...,"[중립, 자극, 이, 무조건, 자극, 과, 반복, 적, 으로, 연합, 되어, 중립,...","[중립, 자극, 무조건, 자극, 반복, 적, 연합, 되어, 중립, 자극, 만으로도,...",고전적 조건 형성


In [7]:
df = df[['questions_cleaned', 'answer_label']]
df.head()

,questions_cleaned,answer_label
0,"[환자, 특정, 행동, 관찰, 하고, 이를, 모방, 하, 거나, 흉내, 내는, 과정...",모델링
1,"[10세, 남아, 약, 2년, 전, 부터, 눈, 자주, 깜박, 거리, 고, 코, 주...",투렛장애
2,"[49세, 여성, 5년, 이상, 불면증, 내, 원하였다, ., 환자, 우울, 불안,...",인지행동치료
3,"[35세, 남자, 갑작스러운, 성격, 변화, 잦은, 돌발, 행동, 병원, 왔다, ....",전두엽
4,"[중립, 자극, 무조건, 자극, 반복, 적, 연합, 되어, 중립, 자극, 만으로도,...",고전적 조건 형성


In [8]:
df.to_csv('output.csv', index=False, encoding='utf-8-sig')